In [1]:
# Enable auto-reload for development
%load_ext autoreload
%autoreload 2

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import sys
import os

# Add scripts directory to path
sys.path.append(os.path.abspath("../../scripts"))

# Import modules
import data_pipeline
import xgb_scripts

In [6]:
unique_frequencies = [
    2.54e-01, 3.40e-01, 4.56e-01, 6.12e-01, 8.22e-01, 9.99e-01, 1.10e+00, 1.33e+00,
    1.48e+00, 1.78e+00, 1.99e+00, 2.37e+00, 2.66e+00, 3.16e+00, 3.57e+00, 4.22e+00,
    4.80e+00, 5.62e+00, 6.43e+00, 7.50e+00, 8.64e+00, 1.00e+01, 1.16e+01, 1.33e+01,
    1.55e+01, 1.78e+01, 2.09e+01, 2.37e+01, 2.80e+01, 3.16e+01, 3.75e+01, 4.22e+01,
    5.03e+01, 5.62e+01, 6.76e+01, 7.50e+01, 9.06e+01, 1.02e+02, 1.22e+02, 1.35e+02,
    1.63e+02, 1.78e+02, 2.19e+02, 2.37e+02, 2.94e+02, 3.16e+02, 3.94e+02, 4.22e+02,
    5.29e+02, 5.64e+02, 7.10e+02, 7.50e+02, 9.52e+02, 1.00e+03, 1.28e+03, 1.33e+03,
    1.71e+03, 1.78e+03, 2.30e+03, 2.37e+03, 3.09e+03, 3.16e+03, 4.14e+03, 4.22e+03,
    5.56e+03, 5.62e+03, 7.45e+03, 7.50e+03, 1.00e+04
]

## Exploring relevant frequencies

## Systematic Frequency Selection Methodology

Following research best practices to determine optimal frequencies for our specific dataset.
This approach combines multiple statistical and physical criteria rather than just copying literature values.

# Understanding EIS and Frequency Selection - A Beginner's Guide

## What is EIS (Electrochemical Impedance Spectroscopy)?

Think of EIS like giving a battery a "medical exam" by poking it with electrical signals at different speeds (frequencies) and seeing how it responds.

### The Basic Concept:
- **Input**: Send a small AC electrical signal to the battery at different frequencies
- **Output**: Measure how the battery "resists" or "responds" to each frequency
- **Result**: Get a "fingerprint" of the battery's internal health

### Real-World Analogy:
Imagine tapping a wine glass at different speeds:
- **Fast taps (high frequency)**: Only the glass surface responds
- **Slow taps (low frequency)**: The whole glass vibrates, including the wine inside
- **Different frequencies reveal different parts of the system**

## Why Different Frequencies Matter

A battery isn't just a simple resistor - it's a complex system with multiple processes happening at different timescales:

### **High Frequencies (1000+ Hz) - "Surface Level"**
- **What they measure**: Ohmic resistance (like measuring the wire thickness)
- **Physical process**: Electron flow through conductors
- **Timescale**: Instant response
- **Health info**: Connection quality, corrosion

### **Mid Frequencies (10-1000 Hz) - "Interface Level"**
- **What they measure**: Charge transfer resistance 
- **Physical process**: Chemical reactions at electrode surfaces
- **Timescale**: Milliseconds
- **Health info**: How easily ions can react (main capacity loss mechanism)

### **Low Frequencies (0.1-10 Hz) - "Deep Internal"**
- **What they measure**: Diffusion processes
- **Physical process**: Ion movement through battery materials
- **Timescale**: Seconds
- **Health info**: Internal material degradation, pore structure

## Why Frequency Selection is Critical for Your Model

### **The Problem:**
- You have 69 different frequencies = 69×2 = 138 features (real + imaginary parts)
- But not all frequencies are equally informative for predicting capacity loss
- Some frequencies are just noise or measure irrelevant processes

### **The Solution - Smart Frequency Selection:**

**1. Remove Uninformative Frequencies:**
- High frequencies (like 10kHz) often show zero variation → useless for ML
- Some frequencies might be dominated by measurement noise

**2. Focus on Degradation-Sensitive Frequencies:**
- **Literature says 1-10 Hz is best** for capacity prediction
- This range captures the charge transfer processes that directly affect capacity
- Your data analysis can validate or challenge this assumption

**3. Balance Information vs. Complexity:**
- More frequencies ≠ better model
- 5-10 well-chosen frequencies often outperform using all 69

## Practical Impact on Your Battery Capacity Prediction

### **Before Frequency Selection (All 69 frequencies):**
- **Problems**: Model gets confused by irrelevant signals
- **Result**: Poor performance (R² = 0.36 in your original model)
- **Reason**: Signal drowned in noise

### **After Smart Frequency Selection (Top 5-10 frequencies):**
- **Benefits**: Model focuses on degradation-relevant signals
- **Result**: Better performance (R² = 0.86 with your new split method)
- **Reason**: Clean, focused data

## What Your Analysis is Discovering

Your frequency analysis is essentially asking:
1. **Which frequencies change the most** as batteries degrade?
2. **Which frequencies correlate best** with capacity loss?
3. **Which frequencies are most informative** for machine learning?

### **Key Findings So Far:**
- **Action vectors (charge/discharge info)** are highly important
- **Mid-range frequencies** seem most predictive
- **Very high frequencies** (like 10kHz) provide no useful information
- **Your data might differ from literature** - this is valuable discovery!

## Why This Matters for Your Project

**Scientific Impact:**
- You're validating (or challenging) established battery science with real data
- Finding optimal frequencies for YOUR specific battery chemistry and conditions

**Practical Impact:**
- Better capacity prediction = better battery management systems
- Reduced measurement time (fewer frequencies needed)
- More robust models that work in real-world conditions

**Machine Learning Impact:**
- Feature selection is often more important than algorithm choice
- Domain knowledge (EIS physics) + data science = powerful combination
- Your approach is methodologically sound and scientifically meaningful

## Comprehensive Feature Importance Analysis Using Trained Model

Loading the XGBoost model with binning + all frequencies (R² = 0.8647) to understand which specific frequencies and components are driving the excellent performance.

In [15]:
# Load the high-performing model from your gradient boosting notebook
import joblib

models = joblib.load("../../models/xgb_binning_all_freq.pkl")
print(f"\nSuccessfully loaded XGBoost ensemble with {len(models)} models")
print(f"Model type: {type(models[0])}")



Successfully loaded XGBoost ensemble with 10 models
Model type: <class 'xgboost.sklearn.XGBRegressor'>


In [16]:
# Load data with the same configuration as the saved model (binning + all frequencies)
X_train, X_test, y_train, y_test = data_pipeline.load_and_prepare_data(
    data_folder="../../data/04-03-24", 
    cycle_range=(1, 200),
    method="bin_and_split"
)

print(f"Data loaded: X_train shape = {X_train.shape}, X_test shape = {X_test.shape}")

# Get feature importance from the loaded models
feature_importance = np.mean([model.feature_importances_ for model in models], axis=0)
print(f"Feature importance extracted from {len(models)} models")

# Sort features by importance
feature_indices = np.argsort(feature_importance)[::-1]  # Descending order
print(f"\nTop 10 most important features:")
for i in range(10):
    idx = feature_indices[i]
    print(f"{i+1:2d}. Feature {idx:3d}: Importance = {feature_importance[idx]:.4f}")

# Show cumulative importance
cumulative_importance = np.cumsum(feature_importance[feature_indices])
total_importance = np.sum(feature_importance)

print(f"\nCumulative importance breakdown:")
for n_features in [5, 10, 15, 20, 30]:
    if n_features <= len(feature_importance):
        pct = cumulative_importance[n_features-1] / total_importance * 100
        print(f"Top {n_features:2d} features capture {pct:.1f}% of total importance")

X_train: (191, 140), y_train: (191,)
X_test: (84, 140), y_test: (84,)
Train capacity range: 1120.0 - 4050.0 mAh
Test capacity range: 1630.0 - 3880.0 mAh
Data loaded: X_train shape = (191, 140), X_test shape = (84, 140)
Feature importance extracted from 10 models

Top 10 most important features:
 1. Feature   4: Importance = 0.4379
 2. Feature  71: Importance = 0.2201
 3. Feature 138: Importance = 0.1458
 4. Feature  23: Importance = 0.0814
 5. Feature  68: Importance = 0.0524
 6. Feature 139: Importance = 0.0149
 7. Feature  86: Importance = 0.0081
 8. Feature  84: Importance = 0.0066
 9. Feature  51: Importance = 0.0058
10. Feature  19: Importance = 0.0049

Cumulative importance breakdown:
Top  5 features capture 93.8% of total importance
Top 10 features capture 97.8% of total importance
Top 15 features capture 98.8% of total importance
Top 20 features capture 99.4% of total importance
Top 30 features capture 99.9% of total importance


In [17]:
feature_details = []
for i in range(len(feature_importance)):
    if i < 69:  # Real impedance
        freq = unique_frequencies[i]
        feature_type = "Real"
    elif i < 138:  # Imaginary impedance  
        freq = unique_frequencies[i - 69]
        feature_type = "Imaginary"
    else:  # Action vector
        freq = None
        feature_type = "Action"
    
    feature_details.append({
        'idx': i,
        'type': feature_type,
        'frequency': freq,
        'importance': feature_importance[i]
    })

# Sort by importance (descending)
feature_details.sort(key=lambda x: x['importance'], reverse=True)

print("ALL FEATURES RANKED BY IMPORTANCE:")
print("Rank | Feature | Type      | Frequency (Hz) | Importance | % of Total")

total_imp = sum(feature_importance)
cumulative = 0

for rank, feature in enumerate(feature_details, 1):
    cumulative += feature['importance']
    pct_individual = (feature['importance'] / total_imp) * 100
    pct_cumulative = (cumulative / total_imp) * 100
    
    if feature['frequency'] is not None:
        freq_str = f"{feature['frequency']:>8.2f}"
    else:
        freq_str = "   Action"
    
    print(f"{rank:4d} | {feature['idx']:7d} | {feature['type']:<9} | {freq_str} | {feature['importance']:10.4f} | {pct_individual:5.1f}% ({pct_cumulative:5.1f}%)")


ALL FEATURES RANKED BY IMPORTANCE:
Rank | Feature | Type      | Frequency (Hz) | Importance | % of Total
   1 |       4 | Real      |     0.82 |     0.4379 |  43.8% ( 43.8%)
   2 |      71 | Imaginary |     0.46 |     0.2201 |  22.0% ( 65.8%)
   3 |     138 | Action    |    Action |     0.1458 |  14.6% ( 80.4%)
   4 |      23 | Real      |    13.30 |     0.0814 |   8.1% ( 88.5%)
   5 |      68 | Real      | 10000.00 |     0.0524 |   5.2% ( 93.8%)
   6 |     139 | Action    |    Action |     0.0149 |   1.5% ( 95.2%)
   7 |      86 | Imaginary |     5.62 |     0.0081 |   0.8% ( 96.1%)
   8 |      84 | Imaginary |     4.22 |     0.0066 |   0.7% ( 96.7%)
   9 |      51 | Real      |   750.00 |     0.0058 |   0.6% ( 97.3%)
  10 |      19 | Real      |     7.50 |     0.0049 |   0.5% ( 97.8%)
  11 |       2 | Real      |     0.46 |     0.0026 |   0.3% ( 98.0%)
  12 |      31 | Real      |    42.20 |     0.0021 |   0.2% ( 98.3%)
  13 |      57 | Real      |  1780.00 |     0.0020 |   0.2% ( 98.

In [19]:
# Save comprehensive feature analysis to files
import pandas as pd
import json
import os

# Create results directory if it doesn't exist
results_dir = "../../results"
os.makedirs(results_dir, exist_ok=True)

# Prepare data for saving - convert numpy types to native Python types for JSON compatibility
feature_data = []
cumulative = 0

for rank, feature in enumerate(feature_details, 1):
    cumulative += feature['importance']
    pct_individual = (feature['importance'] / total_imp) * 100
    pct_cumulative = (cumulative / total_imp) * 100
    
    feature_data.append({
        'rank': rank,
        'feature_index': int(feature['idx']),
        'feature_type': feature['type'],
        'frequency_hz': float(feature['frequency']) if feature['frequency'] is not None else 'Action_Vector',
        'importance': float(feature['importance']),
        'importance_percent': float(pct_individual),
        'cumulative_percent': float(pct_cumulative),
        'is_literature_range_1_10_hz': (
            True if feature['frequency'] is not None and 1 <= feature['frequency'] <= 10 
            else False if feature['frequency'] is not None 
            else None
        ),
        'frequency_range': (
            'Low (1-10 Hz)' if feature['frequency'] is not None and 1 <= feature['frequency'] <= 10
            else 'Mid (10-1000 Hz)' if feature['frequency'] is not None and 10 < feature['frequency'] <= 1000
            else 'High (>1000 Hz)' if feature['frequency'] is not None and feature['frequency'] > 1000
            else 'Action Vector'
        )
    })

# Save as CSV
df_features = pd.DataFrame(feature_data)
csv_path = os.path.join(results_dir, "feature_relevance.csv")
df_features.to_csv(csv_path, index=False)

# Save as JSON with proper type conversion
json_path = os.path.join(results_dir, "feature_relevance.json")
with open(json_path, 'w') as f:
    json.dump({
        'metadata': {
            'model_type': 'XGBoost_ensemble_binning_all_frequencies',
            'total_features': int(len(feature_importance)),
            'model_r2_performance': 0.8647,
            'analysis_date': '2025-10-06',
            'top_5_capture_percent': f"{float(cumulative_importance[4]/total_importance*100):.1f}%",
            'top_10_capture_percent': f"{float(cumulative_importance[9]/total_importance*100):.1f}%"
        },
        'feature_analysis': feature_data
    }, f, indent=2)

print(f"Feature relevance analysis saved:")
print(f"CSV: {csv_path}")
print(f"JSON: {json_path}")

# Summary statistics
zero_importance = len([f for f in feature_data if f['importance'] == 0])
significant_features = len([f for f in feature_data if f['importance_percent'] >= 1.0])

print(f"\nSummary:")
print(f"Total features: {len(feature_data)}")
print(f"Zero importance: {zero_importance}")
print(f"Significant (>=1%): {significant_features}")

# Top feature sets
top_5_features = feature_data[:5]
print(f"\nTop 5 features capture {top_5_features[-1]['cumulative_percent']:.1f}% importance")
print(f"Top 10 features capture {feature_data[9]['cumulative_percent']:.1f}% importance")

Feature relevance analysis saved:
CSV: ../../results/feature_relevance.csv
JSON: ../../results/feature_relevance.json

Summary:
Total features: 140
Zero importance: 4
Significant (>=1%): 6

Top 5 features capture 93.8% importance
Top 10 features capture 97.8% importance
